In [54]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from ITIS_Model_funcs import get_nominal_param,OLS_res,ITIS

In [55]:
param_log,IC = get_nominal_param()
IC = [0.0, 0., 0.1595967, 0., 14.68266827,
          41.49891056, 39.97751646, 11.29827208]

D = '1'         # '1' for ACTH + Cort,
                # '2' for ACTH + Cort + TNF-a
                # '3' for ACTH + Cort + TNF-a + IL10
dpoints = '1'   # '1' for 25, '2' for 13
pickles = ['11', '12', '21', '22', '31', '32']

with open('OLS_Results\\ResAnalysis' + D + dpoints + '.pkl', 'rb') as f:
    results = pickle.load(f)

        ##Reference
        # all_results = {
        #     'output_ids': output_ids,
        #     'param_ids' : param_ids,
        #     'param_in' : param_in,
        #     'twonorms' : twonorms,
        #     't_data' : t_data,
        #     'y_data' : y_data,
        #     'opt_model' : opt_model,
        #     'true_sol' : true_sol,
        #     'param_opt' : param_opt,
        #     'optimal_solution' :np.exp(least_sq_sol.x),
        #     'objvalue' : least_sq_sol.cost
        # }


output_ids = results['output_ids']
true_sol = results['true_sol']
t_data = results['t_data']
param_opt = results['param_opt']
param_ids = results['param_ids']

tstart = 0
dt = .05
t_end =24 + dt
tspace = np.arange(tstart,t_end,dt)
nt = len(t_data)
y_data = np.zeros((len(t_data),len(output_ids)))

##PROBLEM WITH LAST VALUE FOR Y_DATA?
for i, time in enumerate(t_data):
    index = np.where(tspace==time)
    for j,out_id in enumerate(output_ids):
        # This was causing an issue - it gave back a vector of 1s
        y_data[i,j] = true_sol[index,out_id][0][0]

##SENSITIVITY ANALYSIS
h = 1e-6  #amount to perturb parameters
n_param = len(param_opt)
n_states = len(output_ids)

S = np.zeros((n_param, len(t_data) * n_states)) ##Initialize shape of sensitivity matrix.

for i in range(n_param):  #calculate the relative residual sensitivity to each 45 parameters

        param_in = param_log[i]
        param_delta = param_in + h
##RIGHT NOW THIS IS BEING CALCULATED WITH NOISELESS DATA
        S[i, :] = ((1 / h) * (OLS_res(param_delta, y_data, t_data, i, output_ids, param_log, IC)
                                            - OLS_res(param_in, y_data, t_data, i, output_ids, param_log, IC)))

In [56]:
##import rankings from global analysis
with open('paramRankings\\rankingDesign' + D + '.pkl', 'rb') as f:
    results = pickle.load(f)

rank_value = results['rank_value']      # sorted ranking values
param_sorted = results['param_sorted']  #accordingly sorted params as strings
rank_cut = np.array(rank_value)[np.array(rank_value) > .25 * np.array(rank_value)[0]]
param_sorted = np.array(param_sorted)[:len(rank_cut)]

param_titles = ['d1',
'k1','k2','h1','h2','h3','d2',
'k3','k4','h4','d3',
'h5','h6','k5','k6','h7','d4',
'b1','k7','h8','k8','h9','d5','h10',
'b2','k9','k10','k11','d6',
'k12','k13','k14','h11','d7',
'k15','k16','d8',
'alpha', 'k', 'beta', 'L', 'eps', 'delta', 'T', 'Nc']

circadian_param = ['alpha', 'k', 'beta', 'L', 'eps', 'delta', 'T', 'Nc'] #exclude?

cond = 0
unid = []
p_indices = [param_titles.index(param_sorted[0])]
for i in range(1,len(rank_cut)):
    p_indices.append(param_titles.index(param_sorted[i]))
    S_opt = S[p_indices,:].copy()
    F_opt = S_opt@S_opt.T
    cond = np.linalg.cond(F_opt)
    if cond > 1e+5:
        unid.append(p_indices.pop())
        print("Removed: ", unid[-1])


S_opt = S[p_indices,:].copy()
F_opt = S_opt@S_opt.T
C_opt = np.linalg.inv(F_opt)
selected = [param_titles[i] for i in p_indices]
print("Design : ", D + dpoints)
print("Number of selected parameters:", len(p_indices))
print("Selected Parameters: ",  selected)
print(p_indices)
print("Excluded Parameters: ", [param_titles[i] for i in unid])
print("Condition Number of F:", np.linalg.cond(F_opt))
print("Diagonals of C:", np.diag(C_opt))

Design :  11
Number of selected parameters: 15
Selected Parameters:  ['d7', 'h6', 'd8', 'T', 'h11', 'beta', 'k3', 'd4', 'k14', 'h7', 'alpha', 'h4', 'k4', 'd6', 'k6']
[33, 12, 36, 43, 32, 39, 7, 16, 31, 15, 37, 9, 8, 28, 14]
Excluded Parameters:  []
Condition Number of F: 2167.796131262545
Diagonals of C: [0.06446643 0.05527143 0.06039765 0.00527413 0.18144654 0.02578877
 0.08282043 0.10557677 0.03574351 0.20098627 0.05594602 0.08801976
 0.08733297 0.03725133 0.10320911]


In [57]:
#Complete F
F = S@S.T
C = np.linalg.inv(F)
print("Condition Number of F:", np.linalg.cond(F))
print("Diagonals of C:", np.diag(C))
diag = np.diag(C)
unid = [param_titles[i] for i in list(np.where(diag > 1e+3))[0]]
print("Unidentifiable:", unid)

Condition Number of F: 7488860.492243013
Diagonals of C: [ 61.94855074   3.25151554   8.91521696 123.64938071  64.46527142
  43.47665469   4.79195764   2.44448895   4.24192228  12.56403025
  22.68875995  43.74813965  12.61037708   6.77684136  52.96422119
   3.99877358   5.896025     7.95143313  39.67593976  10.83338364
  27.96946128  16.50123968   4.2949885    7.39997017   5.21138451
  37.54095999  11.52725332  18.87034199   1.43192379   8.70727632
  20.32699014   3.32325515   7.36900845   6.75199679   1.60289607
   7.04054683   7.62476104  26.37172008  46.17992049  11.65022881
  31.36059364  18.13130622 102.18392064   1.36878367 107.90594318]
Unidentifiable: []
